# Galdunx RAG Chatbot

This notebook runs the ingestion and chat app for the Galdunx knowledge base.

1. **Run the first code cell** to ingest the knowledge base (loads `knowledge-base/*.md`, chunks, embeds via OpenRouter, stores in ChromaDB).
2. **Run the second code cell** to launch the Gradio chat UI.

**To re-ingest** (e.g. after editing `.md` files): **stop the Gradio app first** (interrupt the kernel or stop the server), then run the first cell again. The database cannot be written while the chat app is using it.

In [ ]:
import sys
from pathlib import Path

folder = Path.cwd()
if not (folder / "ingest.py").exists():
    folder = folder / "week5" / "IbrahimSheriff"
sys.path.insert(0, str(folder))

import ingest

documents = ingest.fetch_documents()
chunks = ingest.create_chunks(documents)
ingest.create_embeddings(chunks)
print("Ingestion complete.")

In [ ]:
import sys
from pathlib import Path

folder = Path.cwd()
if not (folder / "ingest.py").exists():
    folder = folder / "week5" / "IbrahimSheriff"
sys.path.insert(0, str(folder))

import numpy as np
import plotly.graph_objects as go
from sklearn.manifold import TSNE
import chromadb

DB_NAME = str(folder / "vector_db")
client = chromadb.PersistentClient(path=DB_NAME)
collections = client.list_collections()
if not collections:
    raise FileNotFoundError("No vector DB found. Run the ingest cell first.")
coll = collections[0]
data = coll.get(include=["embeddings", "metadatas", "documents"])

embeddings = np.array(data["embeddings"], dtype=np.float32)
n_vectors, n_dims = embeddings.shape
print(f"Loaded {n_vectors} vectors with {n_dims} dimensions")

tsne = TSNE(n_components=2, perplexity=min(15, n_vectors - 1), random_state=42)
coords_2d = tsne.fit_transform(embeddings)
print("t-SNE reduction complete.")

metadatas = data["metadatas"] or []
documents = data["documents"] or []
sources = []
for i in range(n_vectors):
    meta = metadatas[i] if i < len(metadatas) else {}
    s = meta.get("source", f"chunk_{i}")
    if isinstance(s, str) and "/" in s:
        s = s.split("/")[-1]
    sources.append(s)
unique_sources = list(dict.fromkeys(sources))
palette = ["blue", "green", "red", "orange", "purple", "brown", "pink", "gray"]
colors = [palette[unique_sources.index(s) % len(palette)] for s in sources]

# Hover text: source + document snippet (like day2)
hover_texts = [
    f"Source: {s}<br>Text: {(doc or '')[:100]}..." 
    for s, doc in zip(sources, documents)
]

fig = go.Figure(data=[go.Scatter(
    x=coords_2d[:, 0],
    y=coords_2d[:, 1],
    mode="markers",
    marker=dict(size=10, color=colors, opacity=0.8),
    text=hover_texts,
    hoverinfo="text",
)])
fig.update_layout(
    title="Knowledge-base embeddings (2D t-SNE)",
    xaxis_title="t-SNE 1",
    yaxis_title="t-SNE 2",
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40),
)
fig.show()

In [ ]:

import sys
from pathlib import Path

folder = Path.cwd()
if not (folder / "app.py").exists():
    folder = folder / "week5" / "IbrahimSheriff"
sys.path.insert(0, str(folder))

import app
app.main()